# Silver: MGB flood areas

## Purpose

Parse every selected MGB flood-susceptibility polygon once and publish it as a reusable Silver geometry table. Table 14 (`silver_region_flood_exposure`), Table 15 (`silver_project_flood_map`), and their validators read this table. None of them parses `geometry_json` again.

Parsing and spatial work used to run in the same query. The lazy views made Databricks parse every polygon again inside the clip-and-dissolve stage, and the largest MGB geometry field has 34,352,521 characters. That combination ran out of executor memory. Publishing the parsed geometry first keeps each stage within memory, and an unchanged snapshot is never parsed twice.

## Grain

One MGB flood-susceptibility polygon (one Bronze source row) per Bronze snapshot per parse rule version.

## Source authority

| Input | Role |
| --- | --- |
| `01-bronze.flood_susceptibility` | Selected MGB rows: `object_id`, raw susceptibility code, and `geometry_json` |
| `01-bronze.load_log` and `04-validation.dq_results` | Compact upstream safety evidence |

This table does not map codes to levels. The approved mapping stays in `config_mgb_susceptibility_mapping`, so a new mapping version never forces a new parse.

## 1. Confirm upstream source safety

**What:** require the newest terminal MGB load to be `SUCCESS` or `SKIPPED_IDEMPOTENT`, require its latest Bronze validation to have no blocking failure, and require the current Bronze rows to belong to that snapshot.

**Why:** the newest audit event wins, so an older success cannot hide a newer failed load. Only audit metadata and lineage columns are read here, not the large geometry field.

In [0]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `02-silver`;

-- newest terminal audit event for the MGB source, including failed or unfinished loads
CREATE OR REPLACE TEMPORARY VIEW latest_mgb_flood_area_source_load AS
SELECT * EXCEPT (terminal_rank)
FROM (
    SELECT *, ROW_NUMBER() OVER (
        ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
    ) AS terminal_rank
    FROM `01-bronze`.load_log
    WHERE table_name = 'flood_susceptibility'
      AND status IN ('STARTED', 'SUCCESS', 'FAILED', 'SKIPPED_IDEMPOTENT')
)
WHERE terminal_rank = 1;

CREATE OR REPLACE TEMPORARY VIEW latest_mgb_flood_area_bronze_validation AS
SELECT * EXCEPT (validation_rank)
FROM (
    SELECT dq.*, DENSE_RANK() OVER (ORDER BY dq.run_ts DESC, dq.run_id DESC) AS validation_rank
    FROM `04-validation`.dq_results AS dq
    INNER JOIN latest_mgb_flood_area_source_load AS load
      ON dq.table_name = load.table_name
     AND dq.snapshot_id = load.snapshot_id
)
WHERE validation_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) = 1
    AND COUNT_IF(status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')) = 1
    AND COUNT_IF(snapshot_id IS NOT NULL AND TRIM(snapshot_id) <> '') = 1
    AND COUNT_IF(rows_loaded > 0) = 1,
    'STOP: latest MGB Bronze load is missing, failed, unidentified, or empty'
)
FROM latest_mgb_flood_area_source_load;

SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT_IF(action = 'stop' AND status IN ('FAIL', 'ERROR')) = 0,
    'STOP: latest MGB Bronze validation is missing or unsafe'
)
FROM latest_mgb_flood_area_bronze_validation;

-- lineage columns only, so this does not read the large geometry field
SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT _source_snapshot_id) = 1
    AND COUNT(DISTINCT _ingest_run_id) = 1
    AND MAX(_source_snapshot_id) = (SELECT snapshot_id FROM latest_mgb_flood_area_source_load),
    'STOP: current MGB Bronze rows do not match the validated MGB snapshot'
)
FROM `01-bronze`.flood_susceptibility;

## 2. Select the snapshot and the parse rule version

**What:** declare the parse rule version and the source SRID once, select the MGB snapshot, and derive one deterministic run ID. Then prove that every selected row has one unique `object_id`.

**Why:** the run ID hashes the snapshot, ingest run, optional publisher version, and parse rule version. No date, timestamp, UUID, or random value enters logical identity. `object_id` is the MGB `OBJECTID`, so it is the source key of each polygon. Bronze does not check it for this source, so this notebook does, before it builds keys from it.

**When to change `parse_rule_version`:** whenever the parser below changes in a way that can change a geometry or a status. The new version makes the next run parse again.

In [0]:
%sql
-- every rule constant is declared once here and referenced by name below
DECLARE OR REPLACE VARIABLE source_geometry_srid INT DEFAULT 4326;
-- D-43: an OGC-invalid polygon is repaired only when its area stays this close to the MGB Shape__Area
DECLARE OR REPLACE VARIABLE repair_area_tolerance_ratio DOUBLE DEFAULT 0.01;
DECLARE OR REPLACE VARIABLE repair_area_tolerance_sqm DOUBLE DEFAULT 10000;
-- Shape__Area is in Web Mercator square metres, so the repaired area is measured in the same CRS
DECLARE OR REPLACE VARIABLE source_area_srid INT DEFAULT 3857;
DECLARE OR REPLACE VARIABLE parse_rule_version STRING;
SET VAR parse_rule_version = CONCAT(
    'silver-mgb-flood-area-parse-v2|esri-rings-as-text|holes-when-first-reading-invalid',
    '|buffer0-invalid-within-ratio=', repair_area_tolerance_ratio,
    '-or-sqm=', repair_area_tolerance_sqm
);

CREATE OR REPLACE TEMPORARY VIEW selected_mgb_flood_area_version AS
WITH mgb_context AS (
    SELECT
        MAX(_source_snapshot_id) AS mgb_source_snapshot_id,
        MAX(_ingest_run_id) AS mgb_source_ingest_run_id,
        MAX(_ingested_at) AS source_load_ts,
        COUNT(*) AS selected_mgb_rows,
        COUNT_IF(object_id IS NULL OR TRIM(object_id) = '') AS blank_object_id_rows,
        COUNT(*) - COUNT(DISTINCT TRIM(object_id)) AS duplicate_object_id_rows
    FROM `01-bronze`.flood_susceptibility
),
mgb_publisher AS (
    -- optional publisher label recorded by the load that created the selected rows
    SELECT MAX(NULLIF(TRIM(log.source_version), '')) AS mgb_source_version
    FROM `01-bronze`.load_log AS log
    INNER JOIN mgb_context AS context
        ON log.run_id = context.mgb_source_ingest_run_id
    WHERE log.table_name = 'flood_susceptibility'
      AND log.status = 'SUCCESS'
)
SELECT
    mgb_context.*,
    mgb_publisher.mgb_source_version,
    load.rows_loaded AS audited_rows_loaded,
    parse_rule_version AS parse_rule_version,
    SHA2(CONCAT_WS(
        '||',
        mgb_context.mgb_source_snapshot_id, mgb_context.mgb_source_ingest_run_id,
        COALESCE(mgb_publisher.mgb_source_version, 'NO_PUBLISHER_VERSION'),
        parse_rule_version
    ), 256) AS mgb_flood_area_run_id
FROM mgb_context
CROSS JOIN mgb_publisher
CROSS JOIN latest_mgb_flood_area_source_load AS load;

SELECT ASSERT_TRUE(
    COUNT(*) = 1
    AND COUNT_IF(mgb_source_snapshot_id IS NULL OR mgb_source_ingest_run_id IS NULL) = 0
    AND COUNT_IF(selected_mgb_rows <> audited_rows_loaded) = 0,
    'STOP: selected MGB lineage is incomplete or its row count differs from the audited load'
)
FROM selected_mgb_flood_area_version;

SELECT ASSERT_TRUE(
    COUNT_IF(blank_object_id_rows > 0 OR duplicate_object_id_rows > 0) = 0,
    'STOP: MGB object_id is blank or repeated, so it cannot identify one polygon'
)
FROM selected_mgb_flood_area_version;

SELECT * FROM selected_mgb_flood_area_version;

## 3. Skip an unchanged snapshot

**What:** create the table once with an explicit schema, then check whether it already holds the selected run: every row carries the selected `run_id`, and the row count equals the audited Bronze count.

**Why:** parsing the MGB snapshot is the most expensive step in the flood pipeline. When neither the snapshot nor the parse rule changed, the published table is already correct, so the next step keeps it instead of parsing again. A partial or older table is never treated as current.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS `02-silver`.silver_mgb_flood_area (
    mgb_flood_area_key STRING NOT NULL,
    object_id STRING NOT NULL,
    raw_susceptibility_code STRING,
    geometry_wkb BINARY,
    geometry_srid INT,
    geometry_parse_method STRING NOT NULL,
    source_geometry_status STRING NOT NULL,
    mgb_source_snapshot_id STRING NOT NULL,
    mgb_source_ingest_run_id STRING NOT NULL,
    mgb_source_version STRING,
    parse_rule_version STRING NOT NULL,
    run_id STRING NOT NULL,
    source_load_ts TIMESTAMP
)
USING DELTA
COMMENT 'One MGB flood-susceptibility polygon (one Bronze source row) per Bronze snapshot per parse rule version';

DECLARE OR REPLACE VARIABLE mgb_flood_area_is_current BOOLEAN;
SET VAR mgb_flood_area_is_current = (
    SELECT
        published.published_rows = selected.audited_rows_loaded
        AND published.selected_run_rows = published.published_rows
    FROM selected_mgb_flood_area_version AS selected
    CROSS JOIN (
        SELECT
            COUNT(*) AS published_rows,
            COUNT_IF(area.run_id = version.mgb_flood_area_run_id) AS selected_run_rows
        FROM `02-silver`.silver_mgb_flood_area AS area
        CROSS JOIN selected_mgb_flood_area_version AS version
    ) AS published
);

SELECT
    mgb_flood_area_is_current AS is_current,
    CASE WHEN mgb_flood_area_is_current THEN 'SKIP_PARSE' ELSE 'PARSE_AND_PUBLISH' END AS next_action;

## 4. Parse every selected row and publish

**What:** classify every selected MGB row once by geometry status, then publish it with `INSERT OVERWRITE`. The step runs only when Step 3 found the table stale.

**Parser paths:** the parser tries the geometry as published, then the targeted quoted-number normalization, then the Esri `rings` path. A row with a `rings` key skips the two GeoJSON paths, since Esri JSON is never GeoJSON.

**Esri rings (issue #92):** the first reading is built from the `rings` text with string functions, with every ring as its own polygon. Ring arrays are read with `FROM_JSON` only when that reading is OGC-invalid. Then the counterclockwise rings are read as holes and subtracted from the clockwise rings, as the Esri format defines them. A valid first reading is never changed. This follows the source format; it is not a repair.

**Geometry statuses:** `BLANK_GEOMETRY`, `UNPARSEABLE_GEOMETRY`, `EMPTY_GEOMETRY`, `UNSUPPORTED_GEOMETRY_TYPE`, `INVALID_GEOMETRY`, `REPAIRED_GEOMETRY`, and `USABLE_GEOMETRY`. A nonempty, OGC-valid Polygon or MultiPolygon is usable.

**Repair (D-43):** an OGC-invalid polygon is rebuilt with a zero-distance buffer, `ST_BUFFER(geometry, 0)`. Databricks SQL has no `ST_MAKEVALID`. Most MGB invalid polygons have rings that touch or cross themselves, which Esri allows and OGC does not. The repair is kept as `REPAIRED_GEOMETRY` only when the result is a valid, nonempty Polygon or MultiPolygon whose Web Mercator area stays within 1% or 0.01 sq km of the MGB `Shape__Area`. Otherwise the row stays `INVALID_GEOMETRY` with its unrepaired geometry. Valid polygons are never changed. On 2026-10-11 the repair made all 297 invalid polygons valid and changed their total area by 0.02%.

**Storage:** geometry is stored as WKB with its SRID. Readers rebuild it with `ST_GEOMFROMWKB` and `ST_SETSRID`.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW mgb_flood_area_parsed AS
WITH source_rows AS (
    SELECT
        TRIM(mgb.object_id) AS object_id,
        NULLIF(UPPER(TRIM(mgb.flood_susceptibility_code)), '') AS raw_susceptibility_code,
        NULLIF(TRIM(mgb.geometry_json), '') AS geometry_text,
        TRY_CAST(mgb.source_shape_area AS DOUBLE) AS source_shape_area
    FROM `01-bronze`.flood_susceptibility AS mgb
    CROSS JOIN selected_mgb_flood_area_version AS selected
    WHERE mgb._source_snapshot_id = selected.mgb_source_snapshot_id
),
direct_parse AS (
    SELECT
        esri_text.*,
        -- an Esri JSON row is never GeoJSON, so the GeoJSON readers skip it
        CASE
            WHEN esri_text.esri_rings_text IS NULL
            THEN TRY_TO_GEOMETRY(esri_text.geometry_text)
        END AS published_geometry
    FROM (
        SELECT
            source_rows.*,
            -- Esri JSON keeps its coordinates as text under the "rings" key
            NULLIF(REGEXP_EXTRACT(source_rows.geometry_text, '(?s)"rings"\\s*:\\s*(\\[.*\\])', 1), '') AS esri_rings_text
        FROM source_rows
    ) AS esri_text
),
esri_ring_source AS (
    SELECT
        direct_parse.*,
        -- first reading, built as text: every ring is its own polygon.
        -- Text avoids FROM_JSON arrays, which run out of memory on the largest MGB polygons.
        CASE
            WHEN esri_rings_text IS NULL THEN NULL
            WHEN esri_rings_text RLIKE '\\]\\s*\\]\\s*,\\s*\\[\\s*\\['
            THEN TRY_TO_GEOMETRY(CONCAT(
                '{"type":"MultiPolygon","coordinates":[',
                REGEXP_REPLACE(esri_rings_text, '\\]\\s*\\]\\s*,\\s*\\[\\s*\\[', ']]],[[['),
                ']}'
            ))
            ELSE TRY_TO_GEOMETRY(CONCAT('{"type":"Polygon","coordinates":', esri_rings_text, '}'))
        END AS esri_ring_geometry
    FROM direct_parse
),
esri_ring_checked AS (
    SELECT
        esri_ring_source.*,
        esri_ring_geometry IS NOT NULL AND ST_ISVALID(esri_ring_geometry) AS esri_ring_geometry_is_valid
    FROM esri_ring_source
),
esri_ring_arrays AS (
    SELECT
        esri_ring_checked.*,
        -- ring arrays are read only when the first reading is invalid, to find the holes
        CASE
            WHEN esri_rings_text IS NOT NULL AND NOT esri_ring_geometry_is_valid
            THEN FROM_JSON(esri_rings_text, 'ARRAY<ARRAY<ARRAY<DOUBLE>>>')
        END AS esri_rings
    FROM esri_ring_checked
),
esri_ring_parts AS (
    SELECT
        esri_ring_arrays.*,
        -- shoelace signed area per ring: Esri outer rings run clockwise (negative), holes counterclockwise
        TRANSFORM(
            esri_rings,
            ring -> AGGREGATE(
                SEQUENCE(1, SIZE(ring) - 1),
                CAST(0 AS DOUBLE),
                (area, i) -> area + ring[i - 1][0] * ring[i][1] - ring[i][0] * ring[i - 1][1]
            ) / 2
        ) AS esri_ring_signed_areas
    FROM esri_ring_arrays
),
esri_json_parsed AS (
    SELECT
        esri_ring_parts.*,
        CASE
            WHEN esri_ring_parts.esri_rings_text IS NOT NULL
            THEN ST_TRANSFORM(
                ST_SETSRID(
                    CASE
                        -- a valid first reading is kept as it is
                        WHEN esri_ring_geometry_is_valid
                        THEN esri_ring_geometry
                        -- otherwise read the counterclockwise rings as holes in the clockwise rings,
                        -- as the Esri polygon format defines them; this follows the format, it is not a repair
                        WHEN SIZE(FILTER(esri_ring_signed_areas, area -> area < 0)) > 0
                            AND SIZE(FILTER(esri_ring_signed_areas, area -> area > 0)) > 0
                        THEN ST_DIFFERENCE(
                            TRY_TO_GEOMETRY(TO_JSON(NAMED_STRUCT(
                                'type', 'MultiPolygon',
                                'coordinates', TRANSFORM(
                                    FILTER(
                                        ZIP_WITH(esri_rings, esri_ring_signed_areas, (ring, area) -> NAMED_STRUCT('ring', ring, 'area', area)),
                                        part -> part.area < 0
                                    ),
                                    part -> ARRAY(part.ring)
                                )
                            ))),
                            TRY_TO_GEOMETRY(TO_JSON(NAMED_STRUCT(
                                'type', 'MultiPolygon',
                                'coordinates', TRANSFORM(
                                    FILTER(
                                        ZIP_WITH(esri_rings, esri_ring_signed_areas, (ring, area) -> NAMED_STRUCT('ring', ring, 'area', area)),
                                        part -> part.area > 0
                                    ),
                                    part -> ARRAY(part.ring)
                                )
                            )))
                        )
                        ELSE esri_ring_geometry
                    END,
                    3857
                ),
                4326
            )
            ELSE NULL
        END AS esri_geometry
    FROM esri_ring_parts
),
normalized AS (
    SELECT
        esri_json_parsed.*,
        CASE
            WHEN esri_rings_text IS NULL
            THEN TRY_TO_GEOMETRY(REGEXP_REPLACE(
                REGEXP_REPLACE(geometry_text, '\"(\\[[^\"]*\\])\"', CONCAT(CHAR(36), '1')),
                '\"(-?[0-9]+(?:\\.[0-9]+)?(?:[Ee][+-]?[0-9]+)?)\"',
                CONCAT(CHAR(36), '1')
            ))
        END AS normalized_geometry
    FROM esri_json_parsed
),
parsed AS (
    SELECT
        normalized.*,
        COALESCE(
            published_geometry,
            normalized_geometry,
            esri_geometry
        ) AS mgb_geometry
    FROM normalized
),
classified AS (
    SELECT
        parsed.object_id,
        parsed.raw_susceptibility_code,
        parsed.source_shape_area,
        parsed.mgb_geometry,
        CASE
            WHEN parsed.geometry_text IS NULL THEN 'NOT_PARSED'
            WHEN parsed.published_geometry IS NOT NULL THEN 'GEOJSON_AS_PUBLISHED'
            WHEN parsed.normalized_geometry IS NOT NULL THEN 'GEOJSON_QUOTED_NUMBERS_NORMALIZED'
            WHEN parsed.esri_geometry IS NOT NULL THEN 'ESRI_JSON_CONVERTED'
            ELSE 'PARSE_FAILED'
        END AS geometry_parse_method,
        CASE
            WHEN parsed.geometry_text IS NULL THEN 'BLANK_GEOMETRY'
            WHEN parsed.mgb_geometry IS NULL THEN 'UNPARSEABLE_GEOMETRY'
            WHEN ST_ISEMPTY(parsed.mgb_geometry) THEN 'EMPTY_GEOMETRY'
            WHEN REGEXP_REPLACE(UPPER(ST_GEOMETRYTYPE(parsed.mgb_geometry)), '^ST_', '')
                NOT IN ('POLYGON', 'MULTIPOLYGON') THEN 'UNSUPPORTED_GEOMETRY_TYPE'
            WHEN NOT ST_ISVALID(parsed.mgb_geometry) THEN 'INVALID_GEOMETRY'
            ELSE 'USABLE_GEOMETRY'
        END AS source_geometry_status
    FROM parsed
),
repair_attempts AS (
    -- D-43: only an OGC-invalid polygon is repaired, with a zero-distance buffer;
    -- every other row keeps its parsed geometry
    SELECT
        classified.*,
        CASE
            WHEN classified.source_geometry_status = 'INVALID_GEOMETRY'
            THEN ST_BUFFER(classified.mgb_geometry, 0)
        END AS repaired_geometry
    FROM classified
),
repair_checked AS (
    SELECT
        repair_attempts.*,
        COALESCE(
            repaired_geometry IS NOT NULL
            AND NOT ST_ISEMPTY(repaired_geometry)
            AND REGEXP_REPLACE(UPPER(ST_GEOMETRYTYPE(repaired_geometry)), '^ST_', '')
                IN ('POLYGON', 'MULTIPOLYGON')
            AND ST_ISVALID(repaired_geometry),
            FALSE
        ) AS repair_is_valid_polygon
    FROM repair_attempts
),
repair_measured AS (
    SELECT
        repair_checked.*,
        CASE
            WHEN repair_is_valid_polygon
            THEN ST_AREA(ST_TRANSFORM(repaired_geometry, source_area_srid))
        END AS repaired_area_sqm
    FROM repair_checked
),
repaired AS (
    -- a repair is kept only when its area stays within the D-43 tolerance of the MGB Shape__Area
    SELECT
        repair_measured.*,
        COALESCE(
            repair_is_valid_polygon
            AND source_shape_area > 0
            AND ABS(repaired_area_sqm - source_shape_area)
                <= GREATEST(repair_area_tolerance_ratio * source_shape_area, repair_area_tolerance_sqm),
            FALSE
        ) AS is_repaired
    FROM repair_measured
),
published_geometry AS (
    SELECT
        object_id,
        raw_susceptibility_code,
        geometry_parse_method,
        CASE WHEN is_repaired THEN repaired_geometry ELSE mgb_geometry END AS mgb_geometry,
        CASE WHEN is_repaired THEN 'REPAIRED_GEOMETRY' ELSE source_geometry_status END
            AS source_geometry_status
    FROM repaired
)
SELECT
    SHA2(CONCAT_WS('||', classified.object_id, selected.mgb_source_snapshot_id, selected.parse_rule_version), 256)
        AS mgb_flood_area_key,
    classified.object_id,
    classified.raw_susceptibility_code,
    ST_ASWKB(classified.mgb_geometry) AS geometry_wkb,
    ST_SRID(classified.mgb_geometry) AS geometry_srid,
    classified.geometry_parse_method,
    classified.source_geometry_status,
    selected.mgb_source_snapshot_id,
    selected.mgb_source_ingest_run_id,
    selected.mgb_source_version,
    selected.parse_rule_version,
    selected.mgb_flood_area_run_id AS run_id,
    selected.source_load_ts
FROM published_geometry AS classified
CROSS JOIN selected_mgb_flood_area_version AS selected;

-- the parse runs only when the published table is stale; a current table is kept as it is
DECLARE OR REPLACE VARIABLE mgb_flood_area_publish_sql STRING;
SET VAR mgb_flood_area_publish_sql = CASE
    WHEN mgb_flood_area_is_current
    THEN 'SELECT \'SKIPPED_CURRENT\' AS publish_action'
    ELSE 'INSERT OVERWRITE `02-silver`.silver_mgb_flood_area
        SELECT
            mgb_flood_area_key, object_id, raw_susceptibility_code, geometry_wkb, geometry_srid,
            geometry_parse_method, source_geometry_status, mgb_source_snapshot_id,
            mgb_source_ingest_run_id, mgb_source_version, parse_rule_version, run_id, source_load_ts
        FROM mgb_flood_area_parsed'
END;
EXECUTE IMMEDIATE mgb_flood_area_publish_sql;

## 5. Reconcile the published table

**What:** one grouped scan of the published table proves that it holds the selected run, that every Bronze row appears once, that keys are unique, and that usable and repaired geometry is in the source SRID.

**Why:** this gate runs on a parsed table and a skipped table alike, so a skip can never publish a stale or partial table. The status and parse-method counts are the evidence that Tables 14 and 15 report.

In [0]:
%sql
SELECT
    accounting.*,
    ASSERT_TRUE(
        accounting.published_rows = accounting.audited_rows_loaded
        AND accounting.selected_run_rows = accounting.published_rows
        AND accounting.distinct_object_ids = accounting.published_rows
        AND accounting.distinct_keys = accounting.published_rows,
        'STOP: published MGB flood areas do not reconcile with the selected Bronze snapshot'
    ) AS accounting_gate,
    ASSERT_TRUE(
        accounting.usable_wrong_srid_rows = 0 AND accounting.usable_without_wkb_rows = 0,
        'STOP: usable MGB geometry is missing or not in the expected source SRID'
    ) AS geometry_gate,
    ASSERT_TRUE(
        accounting.geometry_text_rows = 0 OR accounting.usable_rows > 0,
        'STOP: no MGB geometry is usable, so the geometry encoding contract is broken'
    ) AS encoding_gate
FROM (
    SELECT
        COUNT(*) AS published_rows,
        MAX(selected.audited_rows_loaded) AS audited_rows_loaded,
        COUNT_IF(area.run_id = selected.mgb_flood_area_run_id) AS selected_run_rows,
        COUNT(DISTINCT area.object_id) AS distinct_object_ids,
        COUNT(DISTINCT area.mgb_flood_area_key) AS distinct_keys,
        COUNT_IF(area.source_geometry_status IN ('USABLE_GEOMETRY', 'REPAIRED_GEOMETRY')) AS usable_rows,
        COUNT_IF(area.source_geometry_status = 'REPAIRED_GEOMETRY') AS repaired_geometry_rows,
        COUNT_IF(area.source_geometry_status <> 'BLANK_GEOMETRY') AS geometry_text_rows,
        COUNT_IF(area.source_geometry_status IN ('USABLE_GEOMETRY', 'REPAIRED_GEOMETRY')
            AND area.geometry_srid <> source_geometry_srid) AS usable_wrong_srid_rows,
        COUNT_IF(area.source_geometry_status IN ('USABLE_GEOMETRY', 'REPAIRED_GEOMETRY')
            AND area.geometry_wkb IS NULL) AS usable_without_wkb_rows,
        COUNT_IF(area.source_geometry_status = 'BLANK_GEOMETRY') AS blank_geometry_rows,
        COUNT_IF(area.source_geometry_status = 'UNPARSEABLE_GEOMETRY') AS unparseable_geometry_rows,
        COUNT_IF(area.source_geometry_status = 'EMPTY_GEOMETRY') AS empty_geometry_rows,
        COUNT_IF(area.source_geometry_status = 'UNSUPPORTED_GEOMETRY_TYPE') AS unsupported_geometry_rows,
        COUNT_IF(area.source_geometry_status = 'INVALID_GEOMETRY') AS invalid_geometry_rows,
        COUNT_IF(area.geometry_parse_method = 'GEOJSON_AS_PUBLISHED') AS parsed_as_published_rows,
        COUNT_IF(area.geometry_parse_method = 'GEOJSON_QUOTED_NUMBERS_NORMALIZED') AS parsed_after_normalization_rows,
        COUNT_IF(area.geometry_parse_method = 'ESRI_JSON_CONVERTED') AS esri_json_converted_rows
    FROM `02-silver`.silver_mgb_flood_area AS area
    CROSS JOIN selected_mgb_flood_area_version AS selected
) AS accounting;

# Summary

- **Table:** `02-silver.silver_mgb_flood_area`.
- **Grain:** one MGB flood-susceptibility polygon (one Bronze source row) per Bronze snapshot per parse rule version.
- **Primary key:** `mgb_flood_area_key`, a SHA-256 hash of `object_id`, the MGB snapshot, and `parse_rule_version`.
- **Uniqueness:** `object_id`, `mgb_source_snapshot_id`, and `parse_rule_version` together.
- **Geometry:** WKB in `geometry_wkb` with `geometry_srid` 4326. Every row keeps its `source_geometry_status` and `geometry_parse_method`. An invalid polygon is repaired only under D-43 and is then marked `REPAIRED_GEOMETRY`; Tables 14 and 15 use it as usable geometry.
- **No mapping here:** raw codes stay raw. Tables 14 and 15 apply the approved mapping.
- **Skip rule:** an unchanged snapshot and parse rule version are not parsed again. Step 5 reconciles the table on every run.
- **Readers:** `09_silver_region_flood_exposure`, `10_silver_project_flood_map`, and their validators `07` and `08`. Validator `06a` checks this table.
- **Runtime:** requires Databricks execution with native spatial SQL and `EXECUTE IMMEDIATE`. No runtime result is claimed by this notebook text.